In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-pulse-vision")

In [2]:
!git clone -b arch-exp --single-branch --depth 1 https://{secret_value_0}@github.com/mmmishaa/pulse-vision.git

Cloning into 'pulse-vision'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 0), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 11.73 KiB | 5.86 MiB/s, done.


In [3]:
import sys
sys.path.append("/kaggle/working/pulse-vision")

In [4]:
from src.data.dataset import RPPGDataset, data_split
from src.models.model import EfficientPhys_C
from src.models.losses import NegativePearsonCorrelation
from src.train.trainer import Trainer

import torch
from torch.utils.data import DataLoader
from pathlib import Path

```json
CONFIG = {
    "clip_len": 100,
    "stride": 200,
    "batch_size": 16,
    "val_ratio": 0.2,
    "seed": 42,
    "lr": 1e-3,
    "betas": (0.9, 0.999),
    "weight_decay": 1e-4,
    "epochs": 20,
}
```

In [5]:
MCD_DIR = Path("/kaggle/input/datasets/mishasavinov/mcd-rppg-dataset")

metadata_path = MCD_DIR / "metadata.csv"
val_df_patients, train_df_patients = data_split(metadata_path=metadata_path)

train_dataset = RPPGDataset(data_path=MCD_DIR, is_train=True, df_patients=train_df_patients)
val_dataset = RPPGDataset(data_path=MCD_DIR, is_train=False, df_patients=val_df_patients)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
model = EfficientPhys_C(n_frames=100).to(device)
criterion = NegativePearsonCorrelation()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

epochs = 20

lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, eta_min=1e-5)

In [8]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    save_dir=Path("/kaggle/working/weights"),
    model_name="efficientphys_c",
    lr_scheduler=lr_scheduler
)

In [9]:
trainer.fit(epochs=epochs)

Training:   5%|▌         | 1/20 [24:06<7:38:05, 1446.61s/it]

Epoch 1: Train Loss: 0.53137, Val Loss: 0.50851


Training:  10%|█         | 2/20 [46:48<6:59:02, 1396.79s/it]

Epoch 2: Train Loss: 0.48518, Val Loss: 0.46282


Training:  15%|█▌        | 3/20 [1:08:16<6:21:43, 1347.27s/it]

Epoch 3: Train Loss: 0.47091, Val Loss: 0.44615


... (30 outputs left out)

Training:  95%|█████████▌| 19/20 [7:14:19<23:36, 1416.65s/it]

Epoch 19: Train Loss: 0.40807, Val Loss: 0.40109


Training: 100%|██████████| 20/20 [7:37:56<00:00, 1373.84s/it]

Epoch 20: Train Loss: 0.40740, Val Loss: 0.40199
Training complete. Weights from epoch 19 loaded. Val Loss: 0.40109
